# 교차 인코더로 관련 논문 후보를 재정렬 (Colab / RunPod)

**하는 일은 하나입니다** — 저장소가 굳혀 보낸 (시드, 후보) 쌍 68,182개를 교차 인코더로
채점해 점수 파일 하나를 만듭니다. **판정은 여기서 하지 않습니다.**

## 왜 교차 인코더인가

[ADR-049](https://github.com/rndigkwk/arxiv_graph_RAG/blob/main/docs/adr/ADR-049-no-local-embedding-clears-the-bar.md)가
로컬 **양방향 인코더(bi-encoder)** 셋을 기각했습니다 — bge-base·bge-m3·Qwen3-0.6B 전부
`text-embedding-3-large`에 구간이 0을 제외하고 졌고, **크기가 아무것도 사지 않았습니다**
(5.2배는 더 나빴고, 5.5배는 1/5.5 크기와 구별되지 않았습니다).

교차 인코더는 **다른 종류**입니다. bi-encoder는 두 논문을 따로 벡터로 만들어 코사인을 재고,
교차 인코더는 **둘을 함께 읽어** 점수를 냅니다. 간선이 없는 논문에서 코사인을 이길 **기전이
있는** 유일한 종류입니다.

## 넘어야 하는 바

미래 라벨 과제의 **간선 0개 집단**(1,112 시드, 그래프가 0점이라 임베딩이 유일한 신호)에서
`vector` 변형(코사인)을 **구간이 0을 제외하도록** 이겨야 합니다.

| 그 집단의 현재 값 | |
|---|---:|
| recall@5 | 0.1227 |
| hit@5 | 0.4739 |
| MRR | 0.3457 |

**재기 전에 적어 둡니다.** 결과를 보고 바를 움직이지 않기 위해서입니다(ADR-039가 그렇게 해서
BGE-M3·Qwen 판정이 성립했습니다).

## 한 번 터졌다면 — **세션을 다시 시작하세요**

GPU 커널이 `cudaErrorAssert`로 죽으면 **그 프로세스의 CUDA가 끝까지 망가집니다.** 설정을
고치고 셀만 다시 돌려도 안 됩니다 — 다음 실행은 보통 **모델을 올리다** 죽고, 그래서 모델이
원인처럼 보입니다.

> 런타임 > **세션 다시 시작** → 1절부터 다시

2절이 작은 CUDA 연산 하나로 그 상태를 **몇 초 만에** 가려내 이 안내를 띄웁니다.

## 이 노트북이 하지 않는 것

**후보를 다시 만들지 않습니다.** `벡터 상위 50 ∪ 보이는 공동인용`이고 저장소의
`pool_candidates`가 계산해 보냈습니다. 여기서 다시 만들면 `graph-text-gbm`과 **다른 후보
집합**을 재게 되고, 그러면 "재정렬이 못 했다"와 "후보가 달랐다"를 구별할 수 없습니다(ADR-029).

**텍스트를 다시 만들지 않습니다.** 굳어 있는 말뭉치의 문자열을 그대로 보냈습니다. 다시 만들면
재는 것이 모델이 아니라 **전처리 차이**입니다(ADR-039).

**순위를 매기지 않습니다.** 점수만 돌려주면 저장소가 정렬하고 쌍체 부트스트랩으로 집단별
구간을 냅니다 — 동점 처리까지 기존 변형들과 같아야 합니다.

## 1. 설정 — 여기만 바꿉니다

In [ ]:
# 이 셀만 바꾸고 나머지는 그대로 실행하세요.
# **이 모델이어야 합니다.** 같은 계열 셋 중 1024를 받는 것은 v2-m3뿐입니다
# (2026-10-20 실측, AutoConfig.max_position_embeddings):
#   bge-reranker-base    512   <- 1024를 주면 장치 어서션으로 터집니다
#   bge-reranker-large   512   <- 큰데도 512입니다
#   bge-reranker-v2-m3  8192   <- 이것
MODEL = "BAAI/bge-reranker-v2-m3"

# 쌍 하나는 (시드 제목+초록, 후보 제목+초록)이고 **둘을 합쳐** 자릅니다. 이 말뭉치의
# 쌍 길이를 모델 토크나이저로 실측하니 중앙값 **636**, p95 850, 최대 1025였습니다.
#   512로 자르면 쌍의 **84.8%**가 잘립니다 -> 재는 것이 모델이 아니라 자른 자리가 됩니다
#   1024면 초과가 **0.2%**입니다
MAX_SEQ = 1024

# 24층·1024은닉 모델을 1024토큰으로 돌립니다. 임베딩 노트북(12층, 512토큰)의 배치를
# 그대로 가져오면 안 됩니다. **32로 두고 첫 조각의 VRAM 피크를 보고** 올리세요 -
# 어림하지 말고 측정으로. OOM이 나면 자동으로 절반씩 줄입니다(터지는 대신 느려집니다).
# 여유가 얼마나 남는지의 실측 하나: 2026-10-22에 256까지 올려 돌았습니다. 어느 GPU
# 였는지는 기록이 없으므로 **그 값을 기본값으로 두지 않습니다** - 위 지시대로 32에서
# 시작해 자기 장치의 피크를 보고 올리세요.
BATCH = 32

# 저장소가 보낸 파일 둘. **HF 데이터셋에 이미 올라가 있습니다** - 그대로 두면 받아 옵니다.
# 파일 창에 직접 올리실 거면 USE_HF=False로 바꾸세요.
PAIRS = "rerank_pairs.jsonl.gz"
TEXTS = "rerank_texts.jsonl.gz"

USE_HF = True
DATA_REPO = "arxiv-graphrag-colab"

# SMOKE=True면 시드 40개만 채점해 코드가 도는지 봅니다.
SMOKE = False


## 2. 환경 확인

GPU가 없으면 멈춥니다. 교차 인코더는 쌍 68,182개에 전체 어텐션을 돌리므로 CPU로는 하루를
넘깁니다.

In [ ]:
try:
    import torch
except ImportError:
    torch = None

if torch is None or not torch.cuda.is_available():
    print('GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 고르세요.')
    if not SMOKE:
        raise SystemExit('GPU 없이 전체 채점을 시작하지 않습니다.')
else:
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'{name}  {total:.1f}GB')

    # **CUDA가 이미 망가져 있지 않은지 먼저 봅니다.** 장치 어서션(cudaErrorAssert)이 한 번
    # 나면 **그 프로세스의 CUDA 컨텍스트가 끝까지 망가집니다.** 그 뒤의 모든 CUDA 호출이
    # 실패하는데, 보통 모델 로드(`.to(device)`)에서 터져서 **원인처럼 보입니다.**
    # 2026-10-20에 정확히 그 일이 있었습니다 - 앞 실행의 길이 초과가 컨텍스트를 깨뜨렸고,
    # 다음 실행은 모델을 올리다 죽었습니다. 작은 연산 하나로 몇 초 만에 가려냅니다.
    try:
        torch.zeros(8, device='cuda').sum().item()
        print('CUDA 상태 정상')
    except Exception as exc:
        raise SystemExit(
            f'CUDA가 이미 망가진 상태입니다({type(exc).__name__}). 앞 실행에서 장치 '
            '어서션이 났을 때 이렇게 됩니다. **런타임 > 세션 다시 시작**을 누르고 1절부터 '
            f'다시 돌리세요. 원문: {str(exc).splitlines()[0][:150]}'
        ) from None

    # 교차 인코더는 쌍 길이가 MAX_SEQ이고 어텐션 활성값이 seq_len^2이라 표본당 무겁습니다.
    # 어림하지 않고 조각마다 실측을 찍습니다 - 2026-10-20에 어림이 2.2배 틀렸습니다.
    print(f'배치 {BATCH}, 최대 길이 {MAX_SEQ}로 시작합니다. OOM이 나면 절반씩 줄입니다.')

In [ ]:
%pip -q install -U sentence-transformers

# 설치만으로 끝내지 않고 **실제로 가져와 봅니다.** 2026-10-20에 huggingface_hub를 따로
# 올렸다가 sentence-transformers와 충돌했고, pip은 경고만 찍고 넘어갔습니다. 그러면 몇 셀
# 뒤 채점에서야 깨집니다.
try:
    import sentence_transformers
    from sentence_transformers import CrossEncoder  # noqa: F401
    print('sentence-transformers', sentence_transformers.__version__)
    print('가져오기 확인 완료')
except Exception as exc:
    raise SystemExit(
        f'{type(exc).__name__}: {exc}\n\n'
        '의존성이 어긋났습니다. 런타임 > 세션 다시 시작을 누르고 이 셀부터 다시 돌리세요. '
        '`huggingface_hub`를 따로 올리지 마세요.'
    )

### 길이 한도 확인 — 가장 먼저 막는 것

**2026-10-20에 여기서 터졌습니다.** `MAX_SEQ = 1024`로 `bge-reranker-base`를 돌렸는데, 그
모델은 위치 임베딩이 **514(실사용 512)** 입니다. 범위를 벗어난 인덱스를 조회해 GPU 커널이
`cudaErrorAssert`로 죽었고, 메시지에는 길이 이야기가 한 줄도 없었습니다.

그래서 **가중치를 내려받기 전에** 설정만 읽어 확인합니다. 설정 파일은 몇 KB이고, 틀렸다면
2.3GB를 받을 이유가 없습니다.

In [ ]:
from transformers import AutoConfig

_cfg = AutoConfig.from_pretrained(MODEL)
_limit = _cfg.max_position_embeddings - 2   # 시작·끝 토큰 몫
print(f'{MODEL}  {_cfg.model_type}  층 {_cfg.num_hidden_layers}  '
      f'위치 {_cfg.max_position_embeddings} -> 최대 길이 {_limit}')

if MAX_SEQ > _limit:
    raise SystemExit(
        f'MAX_SEQ({MAX_SEQ})가 이 모델의 한도({_limit})를 넘습니다. 그대로 돌리면 GPU가 '
        f'cudaErrorAssert로 죽습니다(길이 이야기는 메시지에 나오지 않습니다). '
        f'MAX_SEQ를 {_limit} 이하로 내리거나, 1024를 받는 모델을 고르세요 '
        '(BAAI/bge-reranker-v2-m3는 8192까지 받습니다).'
    )
print(f'MAX_SEQ {MAX_SEQ} <= {_limit} 확인')

## 3. 허깅페이스 토큰 — **여기서 한 번만** 묻습니다

이 노트북에서 **입력을 기다리는 유일한 곳**이고, 긴 채점보다 **먼저** 둡니다. 전에는 6절에서
물었는데, 그러면 한 시간 채점이 끝난 뒤에 입력을 기다리며 멈춰 있게 됩니다.

쌍 파일을 받을 때(4절)와 점수를 올릴 때(7절) **같은 토큰을 재사용합니다.**

- **write 권한**이 필요합니다(6절이 올립니다).
- 화면에 찍히지 않습니다(`getpass`).
- Colab 시크릿에 `HF_TOKEN`이 있으면 묻지 않고 그걸 씁니다 — 다만 VS Code로 붙은 세션에서는
  시크릿 조회가 시간 초과되므로 그때는 직접 입력받습니다.

In [ ]:
import getpass

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        print('Colab 시크릿의 HF_TOKEN을 씁니다.')
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('HF 토큰(write 권한 필요): ')

if not HF_TOKEN:
    raise SystemExit('토큰이 비었습니다. 4절의 받기와 7절의 올리기에 필요합니다.')

# 누구의 토큰인지만 확인합니다. 토큰 자체는 출력하지 않습니다.
from huggingface_hub import HfApi

HF_OWNER = HfApi(token=HF_TOKEN).whoami()['name']
print('계정:', HF_OWNER)

## 4. 쌍 받기

저장소에서 `uv run python -m evals.export_rerank_pairs`로 만든 파일 둘입니다.

| 파일 | 내용 | 크기 |
|---|---|---:|
| `rerank_pairs.jsonl.gz` | 줄마다 `{seed, candidates: [id, ...]}` | 0.3MB |
| `rerank_texts.jsonl.gz` | 쌍에 등장하는 논문 24,360편의 `{arxiv_id, text}` | 12MB |

**후보 순서가 그 파일에서 굳었습니다.** 정렬하거나 다시 뽑지 않습니다.

In [ ]:
import gzip
import json
from pathlib import Path


def fetch(name):
    # 3절에서 받은 토큰을 재사용합니다. 여기서 다시 묻지 않습니다.
    path = Path(name)
    if path.exists():
        return path
    if not USE_HF:
        raise SystemExit(f'{name}이 없습니다. 왼쪽 파일 창에 올리거나 USE_HF=True로 두세요.')
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(f'{HF_OWNER}/{DATA_REPO}', name,
                                repo_type='dataset', token=HF_TOKEN))


def read_jsonl(path):
    with gzip.open(path, 'rt', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle]


pairs = read_jsonl(fetch(PAIRS))
texts = {row['arxiv_id']: row['text'] for row in read_jsonl(fetch(TEXTS))}

if SMOKE:
    pairs = pairs[:40]

# 텍스트가 없는 쌍이 있으면 멈춥니다. 빈 문자열을 채점하면 그 점수가 모델의 사실처럼
# 보고서에 들어갑니다.
missing = {paper for row in pairs for paper in [row['seed'], *row['candidates']]
           if paper not in texts}
if missing:
    raise SystemExit(f'텍스트가 없는 논문 {len(missing)}편: {sorted(missing)[:5]}')

total_pairs = sum(len(row['candidates']) for row in pairs)
sizes = sorted(len(row['candidates']) for row in pairs)
print(f'시드 {len(pairs)}개, 쌍 {total_pairs}개, 논문 {len(texts)}편')
print(f'후보 수 중앙값 {sizes[len(sizes) // 2]}, 최대 {sizes[-1]}')

## 5. 채점

`CrossEncoder.predict`에 `(시드 텍스트, 후보 텍스트)` 쌍을 넘깁니다. **점수만 받습니다.**

중간에 끊겨도 처음부터 하지 않도록 시드 200개씩 조각으로 저장합니다. 다시 실행하면 남은
조각부터 이어서 합니다.

**OOM이 나면 배치를 절반으로 줄여 다시 시도합니다.** 조각 저장과 합쳐지면 터지는 대신
느려지기만 합니다. 2026-10-20에 Qwen3 인코딩이 배치 128에서 OOM으로 64까지 내려가 완주한
적이 있습니다 — 그 장치가 없었으면 98분이 아니라 0분이었습니다.

In [ ]:
import re
import time

import numpy as np
import torch   # 이 셀만 다시 돌려도 되게 여기서도 가져옵니다(세션을 다시 시작한 경우).
from sentence_transformers import CrossEncoder

slug = re.sub(r'[^a-z0-9]+', '-', MODEL.lower()).strip('-')
out = Path(f'rerank-{slug}.jsonl.gz')
chunks = Path(f'chunks-rerank-{slug}')
chunks.mkdir(exist_ok=True)
STEP = 200

model = CrossEncoder(MODEL, max_length=MAX_SEQ)


def where(cross_encoder):
    # 어느 장치에 올라갔는지. sentence-transformers 판본마다 속성이 다릅니다.
    for attr in ('model', '_model', 'module'):
        inner = getattr(cross_encoder, attr, None)
        if inner is not None and hasattr(inner, 'parameters'):
            return str(next(inner.parameters()).device)
    return str(getattr(cross_encoder, 'device', 'unknown'))


# **어디에 올라갔는지 확인합니다.** 조용히 CPU로 가면 한 시간이 아니라 몇 시간이고,
# 끝나고서야 압니다.
device = where(model)
print('모델', MODEL, '| 최대 길이', MAX_SEQ, '| 장치', device)
if not SMOKE and device.startswith('cpu'):
    raise SystemExit('모델이 CPU에 올라갔습니다. 런타임 유형에서 GPU를 고르고 다시 돌리세요.')


def is_oom(exc):
    # OOM인가. **형으로만 보지 않습니다** - 경로에 따라 평범한 RuntimeError나 다른 형으로
    # 올라오는 경우가 있고, 그것을 놓치면 반감 장치가 비껴갑니다.
    oom_types = tuple(
        t for t in (getattr(torch.cuda, 'OutOfMemoryError', None),
                    getattr(torch, 'OutOfMemoryError', None))
        if isinstance(t, type)
    )
    if oom_types and isinstance(exc, oom_types):
        return True
    text = f'{type(exc).__name__}: {exc}'.lower()
    return 'out of memory' in text or 'cuda error' in text or 'cublas' in text


def score(batch, size):
    # OOM이면 배치를 절반으로 줄여 다시 시도합니다. 1까지 줄여도 안 되면 올립니다.
    while True:
        try:
            return model.predict(batch, batch_size=size, show_progress_bar=False), size
        except Exception as exc:
            if not is_oom(exc) or size <= 1:
                raise
            torch.cuda.empty_cache()
            size = max(1, size // 2)
            print(f'  OOM({type(exc).__name__}) - 배치를 {size}로 줄여 다시 시도합니다',
                  flush=True)


start = time.perf_counter()
size = BATCH
for begin in range(0, len(pairs), STEP):
    part = chunks / f'{begin:06d}.jsonl'
    if part.exists():
        continue  # 끊겼다가 다시 돌린 경우
    block = pairs[begin:begin + STEP]
    flat, spans = [], []
    for row in block:
        seed_text = texts[row['seed']]
        spans.append(len(row['candidates']))
        flat.extend((seed_text, texts[paper]) for paper in row['candidates'])
    values, size = score(flat, size)
    values = np.asarray(values, dtype=np.float32)
    with part.open('w', encoding='utf-8') as handle:
        cursor = 0
        for row, span in zip(block, spans):
            handle.write(json.dumps({
                'seed': row['seed'],
                'candidates': row['candidates'],
                'scores': [round(float(v), 6) for v in values[cursor:cursor + span]],
            }, ensure_ascii=False) + '\n')
            cursor += span
    done = min(begin + STEP, len(pairs))
    peak = (torch.cuda.max_memory_allocated() / 1e9
            if torch is not None and torch.cuda.is_available() else 0.0)
    print(f'{done}/{len(pairs)} 시드  {(time.perf_counter() - start) / 60:.1f}분  '
          f'배치 {size}  VRAM 피크 {peak:.1f}GB', flush=True)

scored = []
for part in sorted(chunks.glob('*.jsonl')):
    with part.open(encoding='utf-8') as handle:
        scored.extend(json.loads(line) for line in handle)
print(f'채점한 시드 {len(scored)}개')

## 6. 저장하고 확인

시드 수와 쌍 수가 보낸 것과 다르면 저장하지 않습니다 — 어긋난 점수 파일을 가져가면 **조용히
다른 후보의 점수로** 순위가 매겨집니다. 저장소 쪽도 같은 검사를 합니다.

**점수가 전부 같지 않은지도 봅니다.** 교차 인코더가 잘못 올라가면(헤드가 어긋나면) 모든 쌍에
같은 값을 주는데, 그 파일은 그냥 돌아가고 **점수만 낮아집니다.**

In [ ]:
if len(scored) != len(pairs):
    raise SystemExit(f'시드 수가 다릅니다: {len(scored)} != {len(pairs)}')
if sum(len(row['scores']) for row in scored) != total_pairs:
    raise SystemExit('쌍 수가 다릅니다')
for row in scored:
    if len(row['scores']) != len(row['candidates']):
        raise SystemExit(f'{row["seed"]}의 점수 수가 후보 수와 다릅니다')

flat_scores = np.concatenate([np.asarray(r['scores'], dtype=np.float32) for r in scored])
if not np.isfinite(flat_scores).all():
    raise SystemExit('유한하지 않은 점수가 있습니다')
spread = float(flat_scores.max() - flat_scores.min())
if spread < 1e-6:
    raise SystemExit(f'점수가 전부 같습니다(폭 {spread}). 모델이 올라가지 않았습니다.')

with gzip.open(out, 'wt', encoding='utf-8', newline='\n') as handle:
    for row in scored:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')

card = {
    'model': MODEL,
    'seeds': len(scored),
    'pairs': int(flat_scores.size),
    'max_length': MAX_SEQ,
    'batch_size_used': int(size),
    'score_min': round(float(flat_scores.min()), 6),
    'score_max': round(float(flat_scores.max()), 6),
    'score_median': round(float(np.median(flat_scores)), 6),
    'minutes': round((time.perf_counter() - start) / 60, 1),
    'pairs_file': PAIRS,
    'smoke': SMOKE,
}
Path(out.stem + '.json').write_text(json.dumps(card, ensure_ascii=False, indent=2))
print(json.dumps(card, ensure_ascii=False, indent=2))
print(f'{out}  {out.stat().st_size / 1e6:.1f}MB')

## 7. 가져가기 — HF 데이터셋을 거칩니다

**`google.colab.files.download()`를 쓰지 않습니다.** 2026-10-20에 44분 인코딩이 끝난 뒤 그
호출이 `Javascript object`만 찍고 **예외도 없이 아무것도 내려주지 않았습니다** — VS Code나
RunPod처럼 Colab 브라우저 화면이 없는 세션에는 그 JS를 실행할 곳이 없습니다.

아래 셀이 점수 파일과 카드를 **비공개** HF 데이터셋에 올리고, **받을 명령을 repo까지 채워**
찍습니다. 토큰은 **write 권한**이 필요하고 화면에 찍히지 않습니다.

받은 뒤 저장소에서 판정은 두 명령입니다. **OpenAI도 Aura도 쓰지 않습니다.**

```bash
uv run python -m evals.related_study --cases evals/cases_related_future.json \
    --variants rerank --rerank-scores derived/2026-10-14/rerank-baai-bge-reranker-base.jsonl.gz \
    --output reports/rerank_bge_base.json

uv run python -m evals.embedding_slice_study \
    --baseline reports/embedding_openai.json --candidate reports/rerank_bge_base.json
```

In [ ]:
import re
from pathlib import Path

from huggingface_hub import HfApi

# `out`이 아니라 `MODEL`에서 다시 만들고, 쓰는 것을 여기서 다시 임포트합니다 - 채점이 끝난
# 뒤 커널이 끊겨도 1·3절과 이 셀만으로 올라가야 합니다.
_slug = re.sub(r'[^a-z0-9]+', '-', MODEL.lower()).strip('-')
_targets = [p for p in (Path(f'rerank-{_slug}.jsonl.gz'), Path(f'rerank-{_slug}.json'))
            if p.exists()]
if not _targets:
    raise SystemExit(f'rerank-{_slug}.jsonl.gz 가 없습니다. 6절을 먼저 돌리세요.')
for _p in _targets:
    print(f'{_p.name}  {_p.stat().st_size / 1e6:.1f}MB')

# **토큰을 다시 묻지 않습니다** - 2절에서 받았습니다. 한 시간 채점이 끝난 뒤 입력을
# 기다리며 멈춰 있지 않게 하려는 것이 그 셀을 위로 올린 이유입니다(3절).
_api = HfApi(token=HF_TOKEN)
_repo = f"{HF_OWNER}/{DATA_REPO}"
_api.create_repo(_repo, repo_type='dataset', private=True, exist_ok=True)
for _p in _targets:
    _api.upload_file(path_or_fileobj=str(_p), path_in_repo=_p.name,
                     repo_id=_repo, repo_type='dataset')
    print('올림:', _p.name)

print(f'\nrepo: {_repo}')
print('저장소에서 받을 명령:')
print(f'  hf download {_repo} {" ".join(p.name for p in _targets)} \\')
print('      --repo-type dataset --local-dir derived/2026-10-14')